In [8]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
import os

In [15]:
class DataStore:
  def __init__(self, debug:bool=False, **kwargs):
    # super().__init__(
    #     debug=debug,
    #     **kwargs
    # )
    self.debug = debug

  def download(self, indices:str, start:str, end:str, interval:str):
    data = {}
    for ticker in indices:
        clean_ticker = ticker.strip()
        try:
            tmp_data = yf.Ticker(clean_ticker) \
                          .history(
                              start=start,
                              end=end,
                              interval=interval
                          )["Close"]

            if tmp_data.empty:
                print(f"Warning: No data returned for {clean_ticker}")
                continue
            tmp_data.index = pd.to_datetime(tmp_data.index)\
                                    .tz_localize(None)\
                                    .normalize()
            tmp_data = tmp_data[~tmp_data.index.duplicated(keep='last')]
            data[clean_ticker] = tmp_data
        except Exception as e:
            print(f"Failed to download {clean_ticker}: {e}")

    if not data:
        raise ValueError(
            "No data was successfully downloaded  \
              for any of the provided tickers."
        )
    df = pd.concat(data, axis=1)
    return df.sort_index()

  def get_data(
      self,
      tickers:list,
      start:str,
      end:str,
      interval:str="1d",
      benchmark:str="^GSPC"
  ):
    self.benchmark_ticker = benchmark
    df_path = f"portfolio_{start}_{end}.parquet"
    benchmark_path = f"benchmark_{start}_{end}.parquet"

    if not os.path.exists(df_path) or not os.path.exists(benchmark_path):
      df = self.download(tickers, start, end, interval)
      bench_data = self.download(benchmark, start, end, interval)

      df.to_parquet(df_path)
      bench_data.to_parquet(benchmark_path)

    else:
      df = pd.read_parquet(df_path)
      bench_data= pd.read_parquet(benchmark_path)

    returns_raw = df.pct_change().dropna()
    benchmark = bench_data.pct_change().dropna()

    return returns_raw, benchmark

  def plot_data(self):
    (np.cumsum(self.returns_raw * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

  def plot_benchmark(self):
    (np.cumsum(self.benchmark * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

In [213]:
class HMMClassifier:
  def __init__(self, hmm_featurs):
    pass



  def get_regime_probs(self, returns, eff_dim):
    pass

In [214]:
class RegimeClassifier:
  def __init__(self, hmm_classifier, debug=False, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.hmm_classifier = hmm_classifier
    self.debug = debug

  def z_score_norm(self, X):
    X_cent = X - np.mean(X, axis=0)
    return X_cent / np.std(X_cent, axis=0, ddof=0)

  def _apply_ledoit_wolf(self, r_norm, gram_sample, T, n_assets):
    mean_mkt_corr = (np.sum(gram_sample) - n_assets) / (n_assets*(n_assets-1))
    target = np.full((n_assets, n_assets), mean_mkt_corr)
    np.fill_diagonal(target, 1)

    noise_mtx = 0.0

    for t in range(T):
      r_t = r_norm[t, :].reshape(-1, 1)
      sample_t_mtx = r_t @ r_t.T
      noise_mtx += np.sum((sample_t_mtx - gram_sample)**2)

    noise = np.sum(noise_mtx) / (T**2)
    dist = np.sum((gram_sample-target)**2)

    if dist == 0:
      return gram_sample

    if self.debug:
      print(f"Ledoit-Wolf Shrinkage Intensity (Optimal Lambda): {shrinkage_intensity:.4f}")

    shrinkage_intensity = np.clip(noise/dist, 0.0, 1.0)

    return shrinkage_intensity * target + (1-shrinkage_intensity) * gram_sample

  def calculate_spec_ent(self, eig_vals):
    regime_probs = eig_vals / np.sum(eig_vals)
    spec_ent = -np.sum(regime_probs * np.log(regime_probs))

    return spec_ent

  def _get_eff_dim(self, r_norm, T):
    G = self._get_gram_mtx(r_norm, T)
    n_assets = G.shape[0]

    G_cond = self._apply_ledoit_wolf(r_norm, G, T, n_assets)
    eig_vals = np.clip(np.linalg.eigvalsh(G_cond), 1e-12)

    spec_ent = self.calculate_spec_ent(eig_vals)

    N_eff = np.exp(spec_ent)

    return N_eff

  def _get_gram_mtx(self, r, T):
    G = 1/T * r.T @ r
    return G

  def get_regime(self, returns):
    r_log = np.log(1+returns)
    T = r_log.shape[0]

    r_norm = self.z_score_norm(r_log)
    eff_dim = self._get_eff_dim(r_norm, T)

    regime_prob = self.hmm_classifier.get_regime_probs(r_log, eff_dim)

    return regime_prob

In [174]:
data, benchmark = DataStore().get_data(
    tickers=portfolio_feed,
    start="2018-01-01",
    end="2026-01-01",
    interval="1d",
    benchmark=benchmark_feed
)

In [ ]:
class Optimizer:
  def __init__(self, debug, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug

  def optimize(self, returns, risk_aversion):
    pass




In [ ]:
class PortfolioBuilder(Optimizer, DataStore, GeomRegimeDetector):
  def __init__(self, debug, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug

  def CAPM(self, X, y):
    betas = {}
    for col in y.columns:
      cov = np.cov(X, y[col])
      beta = cov[0, 1] / cov[1, 1]
      betas[col] = beta

    return pd.Series(betas)

  def transform_data(data, benchmark):
    returns = data.pct_change()
    mkt_returns = benchmark.pct_change()

    betas = self.CAPM(mkt_returns, returns)
    regime = self.get_regime(mkt_returns)

    return returns, mkt_returns

